# Sistema Multiagente Autônomo com LLMs para Análise Técnica e de Sentimento na B3

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ehabibn-git/aa-finance/blob/main/notebooks/aa_finance_tcc.ipynb)

**Aluno:** Elias Habib  
**Curso:** Pós-Graduação *GenAI e LLM MASTER* – Pontifícia Universidade Católica do Rio de Janeiro (PUC-Rio)  
**Disciplina:** Projetos de Sistemas Inteligentes de Apoio à Decisão  
**Repositório Oficial:** [github.com/ehabibn-git/aa-finance](https://github.com/ehabibn-git/aa-finance)  

---

## 📌 Sobre esta Prova de Conceito (PoC)
Este notebook executa um **Comitê de Investimentos Multiagente** baseado no framework **CrewAI** e no modelo **Google Gemini**:
1. **Analista Técnico:** Coleta cotações diárias da B3 via `yfinance` e calcula deterministamente no `pandas` as médias móveis (SMA 14, EMA 20) e o Índice de Força Relativa (RSI 14).
2. **Analista de Sentimento:** Rastreia as últimas notícias em português via `duckduckgo_search` e Google News para classificar o sentimento e identificar riscos corporativos/regulatórios.
3. **Diretor de Investimentos (CIO):** Pondera os dois pilares, resolve contradições e delibera a recomendação final de investimento: **COMPRAR**, **VENDER** ou **MANTER**.

### ⚙️ Etapa 1: Instalação e Preparação do Ambiente
Esta célula clona o repositório oficial (caso esteja rodando no Google Colab) e instala todas as dependências necessárias.

In [ ]:
# Clonar repositório e acessar diretório caso execute no Google Colab
import os
import sys

if not os.path.exists("src"):
    print("Clonando repositório oficial da B3...")
    !git clone https://github.com/ehabibn-git/aa-finance.git
    %cd aa-finance

# Garantir que a pasta raiz esteja no path do Python
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

# Instalação silenciosa dos pacotes necessários
!pip install -q -r requirements.txt
print("✅ Ambiente configurado e dependências instaladas com sucesso!")

### 🔑 Etapa 2: Configuração da Chave de API do Google Gemini
> **Dica de Segurança no Google Colab:**  
> Adicione sua chave gratuita nos **Secrets** (ícone de chave 🔑 no menu lateral esquerdo) com o nome `GEMINI_API_KEY` e ative o botão de acesso a este notebook. Caso contrário, o código solicitará que você a digite com segurança.

In [ ]:
import os

gemini_api_key = os.getenv("GEMINI_API_KEY")

# 1. Tentar ler dos Secrets nativos do Google Colab
if not gemini_api_key:
    try:
        from google.colab import userdata
        gemini_api_key = userdata.get("GEMINI_API_KEY")
    except Exception:
        pass

# 2. Se não estiver configurado nos Secrets, solicitar digitação segura
if not gemini_api_key:
    import getpass
    gemini_api_key = getpass.getpass("Digite sua chave GEMINI_API_KEY (ou configure nos Secrets): ")

os.environ["GEMINI_API_KEY"] = gemini_api_key
print("✅ Chave de API do Google Gemini configurada com sucesso!")

### 📊 Etapa 3: Seleção do Ativo da B3
Escolha uma ação da B3 ou digite qualquer outro ticker (ex: `PETR4`, `VALE3`, `ITUB4`, `BBDC4`, `MGLU3`, `WEGE3`).

In [ ]:
#@title Parâmetros de Entrada do Usuário { run: "auto" }
ticker_selecionado = "PETR4" #@param ["PETR4", "VALE3", "ITUB4", "BBDC4", "BBAS3", "MGLU3", "WEGE3", "RENT3", "PRIO3", "SUZB3"] {allow-input: true}
modelo_llm = "gemini/gemini-1.5-flash" #@param ["gemini/gemini-1.5-flash", "gemini/gemini-1.5-pro"]

ticker_ativo = ticker_selecionado.strip().upper()
print(f"🎯 Ativo selecionado para análise: {ticker_ativo}")
print(f"🤖 Modelo de Linguagem escolhido: {modelo_llm}")

### 🔍 Etapa 4: Pré-Visualização das Ferramentas Determinísticas (Tools)
Antes de acionar os agentes, visualizamos os dados brutos calculados pelo Python e as notícias coletadas.  
*Isso comprova para a banca que a matemática foi calculada no processador sem alucinação da LLM.*

In [ ]:
from src.tools.financial_tool import obter_indicadores_tecnicos
from src.tools.news_tool import obter_noticias_mercado
from IPython.display import display, Markdown

# 1. Executar Ferramenta Quantitativa
dados_tecnicos = obter_indicadores_tecnicos(ticker_ativo)
print("=" * 70)
print("📊 INDICADORES TÉCNICOS CALCULADOS (DETERMINÍSTICO VIA PANDAS)")
print("=" * 70)
print(dados_tecnicos.get("resumo_texto", "Erro ao calcular dados."))

print("\n" + "=" * 70)
print("📰 NOTÍCIAS RECENTES COLETADAS EM TEMPO REAL")
print("=" * 70)
# 2. Executar Ferramenta de Notícias
dados_noticias = obter_noticias_mercado(ticker_ativo, max_noticias=4)
for i, n in enumerate(dados_noticias.get("noticias", []), 1):
    print(f"{i}. [{n['fonte']}] {n['titulo']}")
    print(f"   Data: {n['data']}")

### 🚀 Etapa 5: Execução do Comitê de Investimentos Multiagente (CrewAI)
Agora o pipeline sequencial é iniciado:
- O **Analista Técnico** interpreta a tabela de preços e RSI;
- O **Analista de Notícias** lê e classifica as manchetes;
- O **Diretor de Investimentos** delibera o parecer final ponderando os dois lados.

In [ ]:
from src.agents.crew_orchestrator import executar_comite_analise

print("⏳ O comitê de investimentos está reunido e deliberando...")
print("Aguarde cerca de 30 a 60 segundos para a conclusão de todas as tarefas sequenciais.\n")

relatorio_markdown = executar_comite_analise(
    ticker=ticker_ativo,
    modelo=modelo_llm,
    api_key=gemini_api_key
)

print("\n✅ Deliberação concluída!")

### 📋 Etapa 6: Relatório Executivo Final Deliberado

In [ ]:
from IPython.display import display, Markdown

display(Markdown(f"# 📑 Deliberação Oficial para {ticker_ativo}\n\n" + relatorio_markdown))